In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import uuid

spark = SparkSession.builder.getOrCreate()


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
3,application_1764899837815_0004,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
S3_REVIEWS_PATH  = "s3://msba-bigdata-groupproject/data/reviews/reviews.csv"
S3_CUSTOMERS_PATH = "s3://msba-bigdata-groupproject/data/customers.csv"
S3_OUTPUT_PATH    = "s3://msba-bigdata-groupproject/dynamodb/reviews_clean/"
TARGET_ROWS       = 1500000


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
reviews_raw = (
    spark.read
    .option("header", False)
    .option("inferSchema", False)
    .csv(S3_REVIEWS_PATH)
    .toDF("rating", "review_title", "review_text")
)

customers = (
    spark.read
    .option("header", True)
    .csv(S3_CUSTOMERS_PATH)
    .select("customer_id")
)

reviews_raw.show(5)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------+--------------------+--------------------+
|rating|        review_title|         review_text|
+------+--------------------+--------------------+
|     2|Stuning even for ...|This sound track ...|
|     2|The best soundtra...|I'm reading a lot...|
|     2|            Amazing!|"This soundtrack ...|
|     2|Excellent Soundtrack|I truly like this...|
|     2|Remember, Pull Yo...|If you've played ...|
+------+--------------------+--------------------+
only showing top 5 rows

In [6]:
reviews = reviews_raw.limit(TARGET_ROWS)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
def random_uuid():
    return str(uuid.uuid4())

uuid_udf = F.udf(random_uuid)

reviews = reviews.withColumn("product_id", uuid_udf())


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
customer_ids = customers.select("customer_id").collect()
customer_list = [r.customer_id for r in customer_ids]

broadcast_customers = spark.sparkContext.broadcast(customer_list)

@F.udf("string")
def random_customer():
    import random
    return random.choice(broadcast_customers.value)

reviews = reviews.withColumn("customer_id", random_customer())


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
reviews = (
    reviews.withColumn("review_id", uuid_udf())
           .withColumn("review_timestamp", F.current_timestamp())
           .withColumn("verified_purchase", F.lit(0))
           .withColumn("source_system", F.lit("amazon_reviews_traincsv"))
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
def sanitize(col):
    return (
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.regexp_replace(F.col(col), r'[\r\n]+', ' '),
                        '"', "'"
                    ),
                    ',', ';'
                ),
                r'\t', ' '
            ),
            r'[\x00-\x1F\x7F]', ''
        )
    )

reviews = (
    reviews.withColumn("review_title", sanitize("review_title"))
           .withColumn("review_text", sanitize("review_text"))
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
reviews = reviews.filter(F.col("review_id").isNotNull())


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
reviews_final = reviews.select(
    "review_id",
    "customer_id",
    "product_id",
    "rating",
    "review_title",
    "review_text",
    "review_timestamp",
    "verified_purchase",
    "source_system"
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
(
    reviews_final.coalesce(1)
    .write
    .option("header", True)
    .option("quote", "")
    .option("escape", "")
    .mode("overwrite")
    .csv(S3_OUTPUT_PATH)
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…